In [11]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime


c:\Users\HP\Desktop\AG0854__3\testenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
pdf_folder_location = "tesla-annual-reports"

In [13]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [14]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [15]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [16]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [17]:
len(tesla_10k_chunks)

3337

In [19]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\HP\AppData\Local\Temp\ipykernel_24980\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [20]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [12]:
chromadb_client.count_collections()

2

In [21]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [14]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

In [ ]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 5 seconds to avoid rate limiting issues with the vector store

Ensure the library chromadb is latest. Run the below command

In [5]:
import chromadb

from langchain_chroma import Chroma



Below is the api key, endpoint, api version, deployment name for the chat model

 Instantiating the embedding model

Now that the database is uploaded onto the Colab instance, we can unzip it and attach a retriever.

In [22]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [23]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002089259CC50>, search_kwargs={'k': 5})

In [24]:
user_query = "what are the types of vehicles that are planned to release to commercial vehicle market?"

### Through Hypothetical Questions

In [25]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['NVIDIA_API_KEY'] = os.getenv('NVIDIA_API_KEY')

In [26]:
from openai import OpenAI


In [27]:
client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

In [28]:
model_name = "meta/llama-3.1-70b-instruct"

In [32]:

user_message_template = """
<Document>
{document}
</Document>
"""

In [33]:
BATCH_SIZE = 100

chunk_batches = []

for i in range(0, len(tesla_10k_chunks), BATCH_SIZE):

    batch = tesla_10k_chunks[i:i+BATCH_SIZE]

    # Token limit control
    batch_text = "\n\n".join(
        [doc.page_content[:200] for doc in batch]
    )

    chunk_batches.append(batch_text)

print(f"Total batches: {len(chunk_batches)}")


def generate_hypothetical_questions(batch_text):

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": """
You will receive multiple document chunks.

Generate ONLY 3 hypothetical questions that best represent
the information across ALL chunks.

Rules:
- Generate exactly 3 questions.
- One question per line.
- No numbering.
- No explanations.
"""
            },
            {
                "role": "user",
                "content": user_message_template.format(
                    document=batch_text
                )
            }
        ]
    )

    return response.choices[0].message.content


batch_questions = []

for idx, batch_text in enumerate(chunk_batches):

    try:

        questions = generate_hypothetical_questions(batch_text)

        batch_questions.append({
            "batch_id": idx + 1,
            "questions": questions,
            "context": batch_text
        })

        print(f"Processed Batch {idx + 1}")

    except Exception as e:

        print(f"Batch {idx + 1} failed: {e}")

Total batches: 34
Processed Batch 1
Processed Batch 2
Processed Batch 3
Processed Batch 4
Processed Batch 5
Processed Batch 6
Processed Batch 7
Processed Batch 8
Processed Batch 9
Processed Batch 10
Processed Batch 11
Processed Batch 12
Processed Batch 13
Processed Batch 14
Processed Batch 15
Processed Batch 16
Processed Batch 17
Processed Batch 18
Processed Batch 19
Processed Batch 20
Processed Batch 21
Processed Batch 22
Processed Batch 23
Processed Batch 24
Processed Batch 25
Processed Batch 26
Processed Batch 27
Processed Batch 28
Processed Batch 29
Processed Batch 30
Processed Batch 31
Processed Batch 32
Processed Batch 33
Processed Batch 34


In [34]:
batch_questions

[{'batch_id': 1,
  'questions': "What is the composition of Tesla's Board of Directors and their respective backgrounds and qualifications?\nHow does Tesla's compensation philosophy and program align with its mission to accelerate the world's transition to sustainable energy?\nWhat are the key terms and conditions of Elon Musk's 2018 CEO Performance Award, including the vesting and exercise requirements?",
  'context': 'UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t\n\nYes\n\t\t\n☒\n\t\t\t\tNo\t\t\n☐\nIndicate\tby\tcheck\tmark\tif\tthe\tregistrant\tis\tnot\trequired\tto\tfile\treports\tpursuant\tto\tSection\t13\tor\t15(d)\tof\tthe\tAct.\t\t\t\tYes\t\t\n☐\n\t\t\t\t\nNo\n\t\t\n☒\nIndicate\tby\tcheck\tmark\twhether\tthe\tregi\n\nEmerging\tgrowth\tcompany\n\t\n☐\n\t\n\t\n\t\n\t\nIf\tan\temerging\tgrowth

In [35]:
len(batch_questions)

34

In [36]:
import json

questions_only = [
    {
        "batch_id": item["batch_id"],
        "questions": item["questions"]
    }
    for item in batch_questions
]

with open("batch_questions_only.json", "w", encoding="utf-8") as f:
    json.dump(
        questions_only,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Questions saved successfully!")

Questions saved successfully!


In [37]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [38]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

In [39]:
from langchain_core.documents import Document

hypothetical_documents = []

for item in batch_questions:

    hypothetical_documents.append(
        Document(
            page_content=item["questions"],
            metadata={
                "batch_id": item["batch_id"]
            }
        )
    )

In [40]:
print(type(hypothetical_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [41]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_documents)

['1e115c8f-a47b-46cd-8c0c-a8d18dca0337',
 '8a936d33-d955-4f06-9e72-89358962119c',
 '2b6fdeed-3743-47b4-a319-16749e722c07',
 '5978c10f-b6e9-4086-aab0-87783d1f3b88',
 '10265c09-c77d-49fa-b475-8eecbadb5e87',
 '2d5a289c-3b5b-4fee-a388-7b400388a4d5',
 '094d52d8-a5b0-4013-a2cf-97aa8e9be221',
 '2bc30f79-76da-4b43-8135-1306fc898f51',
 'b91d9326-04cf-46e0-a26d-a57a9e4231ea',
 '6d5b149f-f8c1-4060-9723-3abbd81c9149',
 '318e6e7c-98c8-47e2-9c29-22ee6dc9bc7c',
 '7af6447a-b010-48a7-8c88-189d1407c5aa',
 '0da8caa3-23f6-4355-abe3-eb4b66f7b54b',
 '6d44c3a9-3946-4767-a19a-17573c72b770',
 'd1ebd4e3-dfa4-4ecd-9d84-e5a25c72aadb',
 '09d5a530-a780-484d-815d-0f0ee56f3a88',
 'eaf8dcf9-dab2-4d9e-af72-c8c883756e6b',
 '2a89a693-39bf-4e56-b8ee-2b4d41077b57',
 '3b36d789-88a6-4308-a8b4-e6a96d61b50d',
 '78811ace-ed02-4c2c-941e-b3c279170e51',
 '2675bcba-25a8-4689-8317-e0155c14d8f2',
 'ce63ebf2-8edb-4c6b-8f82-bc2f05e5bafb',
 '4b980e69-ef81-4b73-965a-c99f42c1f6c7',
 '06f01c40-75ba-41aa-be45-a97870828717',
 'adc00641-bb8a-

In [42]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [43]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [44]:
hypothetical_questions_retrieved

[Document(id='1e6dbeb9-f944-454e-a150-5f9b7f3de2d7', metadata={'batch_number': 5, 'parent_collection': 'full_document_chunks'}, page_content='What are the main components of the cost of automotive sales revenue?'),
 Document(id='b02149f8-5ff5-40a1-b25f-3fbc1dc50be2', metadata={'batch_number': 23, 'parent_collection': 'full_document_chunks'}, page_content='What are the key factors contributing to the cyclical nature of sales in the automotive industry?'),
 Document(id='7816f2b2-8362-47d5-bf09-61c38620456e', metadata={'batch_number': 23, 'parent_collection': 'full_document_chunks'}, page_content="How does Tesla's target demographic for its vehicles, particularly Model 3 and Model Y, impact its overall sales strategy?"),
 Document(id='cd4c2aa9-a96a-4f93-9acd-1414c962031c', metadata={'batch_number': 10, 'parent_collection': 'full_document_chunks'}, page_content="Will regulatory limitations and industry-wide obstacles hinder Tesla's ability to sell vehicles directly to consumers in various 

In [45]:
print(hypothetical_questions_retrieved[0].metadata)

{'batch_number': 5, 'parent_collection': 'full_document_chunks'}


In [46]:
BATCH_SIZE = 100

retrieved_documents = []

for doc in hypothetical_questions_retrieved:

    batch_num = doc.metadata["batch_number"]

    start_idx = (batch_num - 1) * BATCH_SIZE
    end_idx = min(
        start_idx + BATCH_SIZE,
        len(tesla_10k_chunks)
    )

    retrieved_documents.extend(
        tesla_10k_chunks[start_idx:end_idx]
    )

print(f"Retrieved {len(retrieved_documents)} chunks")

Retrieved 500 chunks


In [47]:
print(retrieved_documents)

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2020-09-30T05:19:35+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf', 'total_pages': 297, 'page': 80, 'page_label': '81'}, page_content='accounted\tfor\tas\tleases,\tno\tlonger\tmeet\tthe\tdefinition\tof\ta\tlease\tand\tare\tinstead\taccounted\tfor\tin\taccordance\twith\tthe\tnew\trevenue\tstandard\n(refer\tto\tthe\t\nLeases\n\tsection\tbelow\tfor\tdetails).\n\t\nCost\tof\tRevenues\nAutomotive\tSegment\nAutomotive\tSales\nCost\tof\tautomotive\tsales\trevenue\tincludes\tdirect\tparts,\tmaterial\tand\tlabor\tcosts,\tmanufacturing\toverhead,\tincluding\tdepreciation\tcosts\tof\ntooling\tand\tmachinery,\tshipping\tand\tlogistic\tcosts,\tvehicle\tconnectivity\tcosts,\tallocations\tof\telectricity\tand\tinfrastructure\tcosts\trelated\tto\tour\nSupercharger\tnetwork,\tand\treserves\tfor\testimated\twarranty\texpenses.\tCost\tof\tautomotive\tsales\trevenues\talso\tin

In [48]:
print(type(retrieved_documents))

<class 'list'>


In [49]:
print(type(retrieved_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [50]:
retrieved_context = "\n\n".join(
    doc.page_content
    for doc in retrieved_documents
)

In [51]:
retrieved_context

'accounted\tfor\tas\tleases,\tno\tlonger\tmeet\tthe\tdefinition\tof\ta\tlease\tand\tare\tinstead\taccounted\tfor\tin\taccordance\twith\tthe\tnew\trevenue\tstandard\n(refer\tto\tthe\t\nLeases\n\tsection\tbelow\tfor\tdetails).\n\t\nCost\tof\tRevenues\nAutomotive\tSegment\nAutomotive\tSales\nCost\tof\tautomotive\tsales\trevenue\tincludes\tdirect\tparts,\tmaterial\tand\tlabor\tcosts,\tmanufacturing\toverhead,\tincluding\tdepreciation\tcosts\tof\ntooling\tand\tmachinery,\tshipping\tand\tlogistic\tcosts,\tvehicle\tconnectivity\tcosts,\tallocations\tof\telectricity\tand\tinfrastructure\tcosts\trelated\tto\tour\nSupercharger\tnetwork,\tand\treserves\tfor\testimated\twarranty\texpenses.\tCost\tof\tautomotive\tsales\trevenues\talso\tincludes\tadjustments\tto\twarranty\texpense\nand\tcharges\tto\twrite\tdown\tthe\tcarrying\tvalue\tof\tour\tinventory\twhen\tit\texceeds\tits\testimated\tnet\trealizable\tvalue\tand\tto\tprovide\tfor\tobsolete\tand\ton-\nhand\tinventory\tin\texcess\tof\tforecasted\td

In [52]:
len(retrieved_context)

652824

In [53]:
qna_system_message1 = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the context.

Rules:
- Use only the provided context.
- Do not make up information.
- If the answer is not available in the context, say:
  "I could not find the answer in the provided context."
- Be concise and accurate.
"""
qna_user_message_template1 = """
Context:
{context}

Question:
{question}

Answer:
"""

final_prompt = [
    {
        "role": "system",
        "content": qna_system_message1
    },
    {
        "role": "user",
        "content": qna_user_message_template1.format(
            context=retrieved_context,
            question=user_query
        )
    }
]

response = client.chat.completions.create(
    model=model_name,
    messages=final_prompt,
    temperature=0
)

final_answer = response.choices[0].message.content
print(final_answer)

Tesla has announced several planned electric vehicles to address additional vehicle markets, including specialized consumer electric vehicles in Cybertruck and the new Tesla Roadster.
